In [33]:
import torch
from torchinfo import summary
from torch.utils.data import DataLoader, random_split
import pandas as pd
import os
from dotenv import load_dotenv
import lightning as L
import sys
from tqdm.notebook import tqdm
import pickle

from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import accuracy_score, f1_score

from src.transformer.transformer_architecture import KeypointTransformer, LitKeypointTransformer
from src.image_processing.mediapipe import MediaPipeProcessor
from src.datasets.ipn_data import IPNData

In [2]:
load_dotenv()
GESTURE_VIDEOS_PATH = os.getenv("IPN_GESTURE_VIDEOS")

In [3]:
# Dataset
df = pd.read_csv(f"{GESTURE_VIDEOS_PATH}/metadata.csv")
df.head()

,Video Name,Frames,Sex,Hand,Background,Illumination,People in Scene,Background Motion,Set,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17
0,1CM1_4_R__229,3751,W,Right,Clutter,Stable,Single,Static,train,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1CM1_4_R__230,3684,W,Right,Clutter,Stable,Single,Static,train,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1CM1_4_R__231,3747,W,Right,Clutter,Stable,Single,Static,train,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1CM1_4_R__232,3858,W,Right,Clutter,Stable,Single,Static,train,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1CM42_11_R__205,3686,M,Right,Plain,Stable,Single,Static,train,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
ds_raw = IPNData()
mpp = MediaPipeProcessor()

In [5]:
label_id_dict = ds_raw.get_label_id_dict()
num_videos = ds_raw.get_num_videos()
video_info_dict = ds_raw.get_video_info()

In [6]:
label_to_index = {}

for k in label_id_dict:
    label, gesture = label_id_dict[k]
    label_to_index[label] = int(k)

In [ ]:
N_V = num_videos
ds = []

for i in tqdm(range(N_V)):
    for j in range(sys.maxsize):
        try:
            frames, label, ID = ds_raw.__getitem__(i, j)

            keypoint_sequence = mpp.process_video(frames)

            ds.append([keypoint_sequence, label_to_index[label]]) # using ID as label
        except:
            break

In [9]:
#Saving new dataset

with open("data/dataset.pkl", "wb") as f:
    pickle.dump(ds, f)

print("Saved!")

Saved!


In [ ]:
N_F = len(video_info_dict)
TEST_F = int(0.1*N_F)
TRAIN_VAL_F = N_F - TEST_F

train_size = int(0.9 * TRAIN_VAL_F)
val_size = TRAIN_VAL_F - train_size

test_ds = ds[TRAIN_VAL_F:]
train_val_ds = ds[:TRAIN_VAL_F]

train_ds, val_ds = random_split(train_val_ds, [train_size, val_size])

In [13]:
def collate_fn(batch):
    sequences, labels = zip(*batch)

    sequences = [torch.tensor(s, dtype=torch.float32) for s in sequences]

    # Padding
    padded_sequences = pad_sequence(sequences, batch_first=True)

    # Masking
    mask = torch.zeros(size=padded_sequences.shape[:2], dtype=torch.bool)
    for(i, s) in enumerate(sequences):
        mask[i, :len(s)] = True

    # Labeling
    labels = torch.tensor([int(l) for l in labels])

    return padded_sequences, mask, labels

In [ ]:
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=32, collate_fn=collate_fn)

input_size = 126
d_model = 132

num_classes = len(label_id_dict)

model = LitKeypointTransformer(input_size=input_size, d_model=d_model, num_classes=num_classes)

trainer = L.Trainer(
    max_epochs=20,
    accelerator="auto",
    devices=1
)


In [15]:
x_batch, mask, y_batch = next(iter(train_loader))
logits = model(x_batch)  
print("x:", x_batch.shape, "y:", y_batch.shape, "logits:", logits.shape)

x: torch.Size([32, 357, 126]) y: torch.Size([32]) logits: torch.Size([32, 14])


In [ ]:
trainer.fit(model, train_loader, val_loader)

In [17]:
from pathlib import Path
save_direct = Path("checkpoints")
trainer.save_checkpoint(save_direct / "keypoint_transformer2.ckpt") 

`weights_only` was not set, defaulting to `False`.


In [35]:
model.eval()

LitKeypointTransformer(
  (model): KeypointTransformer(
    (input_proj): Sequential(
      (0): Linear(in_features=126, out_features=132, bias=True)
    )
    (pos_encode): SinusoidalPositionalEncoding()
    (layers): ModuleList(
      (0-5): 6 x EncodingLayer(
        (attention): MultiHeadAttention(
          (w_q): Linear(in_features=132, out_features=132, bias=True)
          (w_k): Linear(in_features=132, out_features=132, bias=True)
          (w_v): Linear(in_features=132, out_features=132, bias=True)
          (output): Linear(in_features=132, out_features=132, bias=True)
        )
        (norm1): LayerNorm((132,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((132,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
        (ffn): Sequential(
          (0): Linear(in_features=132, out_features=512, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_f

In [ ]:
#Testing model

In [ ]:
all_labels = []
all_preds = []

for i in tqdm(range(len(test_ds))):    
    keypoint_sequence, label = test_ds[i]

    x = torch.tensor(keypoint_sequence, dtype=torch.float32).unsqueeze(0)  # (1, seq_len, 126)

    # prediction
    with torch.no_grad():
        logits = model(x)
        predicted_idx = torch.argmax(logits, dim=-1).item()

    all_labels.append(label)
    all_preds.append(predicted_idx)


In [34]:
# F1 score and accuracy (again)
accuracy = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average="macro")

print(accuracy)
print(f1)

0.774822695035461
0.6186552940873319
